In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torch.utils.data import random_split

In [3]:
train_transforms = transforms.Compose([
    # Random augmentation
    transforms.RandomHorizontalFlip(p=0.15),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=1.5),

    # Preprocess
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],
                        std  = [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    # Preprocess
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],
                        std  = [0.229, 0.224, 0.225])
])

In [4]:
train_full = torchvision.datasets.EuroSAT(root="../data", download=True, transform=train_transforms)
test_full = torchvision.datasets.EuroSAT(root="../data", download=True, transform=test_transforms)

train_size = int(0.8 * len(train_full))
test_size = len(train_full) - train_size

generator = torch.Generator().manual_seed(42)
indicies = torch.randperm(len(train_full), generator=generator).tolist()
train_indicies, test_indicies = indicies[:train_size], indicies[train_size:]

train_set = Subset(train_full, train_indicies)
test_set = Subset(test_full, test_indicies)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

In [5]:
image, label = test_set[0]
image.shape

torch.Size([3, 64, 64])

In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels,):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.block(x)
        return x


class EuroSatClassifier(nn.Module):
    def __init__(self): 
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(3, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device}")

model = EuroSatClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

def train_epoch(model, train_loader, loss_function, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0 

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

        # Track progress
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

        if batch_idx % 100 == 0 and batch_idx > 0:
            accuracy = 100 * correct / total
            print(f"Loss: {running_loss / 100:.3f} | Train accuracy: {accuracy:.1f}")
            running_loss = 0.0


Using mps


In [7]:
def evaluate(model, test_loader, device):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    return 100 * correct / total

In [8]:
num_epoch = 10

for epoch in range(num_epoch):
    print(f"Epoch {epoch}")
    train_epoch(model, train_loader, loss_function, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f"Accuracy: {accuracy}")

Epoch 0
Loss: 0.064 | Train accuracy: 21.4
Loss: 0.027 | Train accuracy: 26.6
Loss: 0.017 | Train accuracy: 30.4
Loss: 0.012 | Train accuracy: 32.9
Loss: 0.009 | Train accuracy: 35.2
Loss: 0.007 | Train accuracy: 37.2
Accuracy: 62.074074074074076
Epoch 1
Loss: 0.041 | Train accuracy: 51.5
Loss: 0.020 | Train accuracy: 51.8
Loss: 0.013 | Train accuracy: 52.5
Loss: 0.009 | Train accuracy: 53.8
Loss: 0.007 | Train accuracy: 54.3
Loss: 0.006 | Train accuracy: 54.8
Accuracy: 72.57407407407408
Epoch 2
Loss: 0.035 | Train accuracy: 58.3
Loss: 0.017 | Train accuracy: 59.0
Loss: 0.011 | Train accuracy: 59.6
Loss: 0.008 | Train accuracy: 60.0
Loss: 0.007 | Train accuracy: 60.6
Loss: 0.005 | Train accuracy: 61.1
Accuracy: 75.0925925925926
Epoch 3
Loss: 0.032 | Train accuracy: 63.0
Loss: 0.016 | Train accuracy: 62.5
Loss: 0.010 | Train accuracy: 63.6
Loss: 0.007 | Train accuracy: 64.0
Loss: 0.006 | Train accuracy: 64.0
Loss: 0.005 | Train accuracy: 64.2
Accuracy: 76.85185185185185
Epoch 4
Loss: 0.